In [184]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_excel("aapl_03152025_04152026.xlsx")

df=df[df['volume']!=0]
df.reset_index(drop=True, inplace=True)
df.isna().sum()
df.tail()


Saving aapl_03152025_04152026.xlsx to aapl_03152025_04152026 (11).xlsx


,Local time,open,high,low,close,volume
266,2026-04-08,258.450,259.7499,256.5300,258.90,41032772
267,2026-04-09,259.000,261.1200,256.0700,260.49,28121574
268,2026-04-10,259.980,262.1900,259.0231,260.48,31291473
269,2026-04-13,259.730,260.1800,256.6600,259.20,36234698
270,2026-04-14,259.245,261.9300,257.1900,258.83,48370710


In [185]:
def calculate_atr(df, period=14):
    high = df['high']
    low = df['low']
    close = df['close']

    tr = pd.Series(index=df.index, dtype=float)

    tr.iloc[0] = high.iloc[0] - low.iloc[0]
    for i in range(1, len(df)):
        tr.iloc[i] = max(
            high.iloc[i] - low.iloc[i],
            abs(high.iloc[i] - close.iloc[i-1]),
            abs(low.iloc[i] - close.iloc[i-1])
        )

    atr = tr.rolling(period).mean()
    return atr

In [186]:
import os

df['ATR'] = calculate_atr(df, period=14)
curr_dir = os.getcwd()
df.to_excel(os.path.join(curr_dir, "ATR.xlsx"), index=False)

In [187]:
# Support and Resistance FUNCTIONS
def support(df1, l, n1, n2): #n1 n2 before and after candle l
    for i in range(l-n1+1, l+1):
        if(df1.low[i]>df1.low[i-1]):
            return 0
    for i in range(l+1,l+n2+1):
        if(df1.low[i]<df1.low[i-1]):
            return 0
    return 1

def resistance(df1, l, n1, n2): #n1 n2 before and after candle l
    for i in range(l-n1+1, l+1):
        if(df1.high[i]<df1.high[i-1]):
            return 0
    for i in range(l+1,l+n2+1):
        if(df1.high[i]>df1.high[i-1]):
            return 0
    return 1


In [188]:
length = len(df)
high = list(df['high'])
low = list(df['low'])
close = list(df['close'])
open = list(df['open'])
atr = list(df['ATR'])

bodydiff = [0] * length
highdiff = [0] * length
lowdiff = [0] * length
ratio1 = [0] * length
ratio2 = [0] * length

def isEngulfing(l):
    bodydiff[l] = abs(open[l] - close[l])

    if bodydiff[l] < 0.1 * atr[l]:
        bodydiff[l] = 0.1 * atr[l]

    bodydiffmin = 0.35 * atr[l]

    # Bearish engulfing
    if (bodydiff[l] > bodydiffmin and
        bodydiff[l-1] > bodydiffmin and
        open[l-1] < close[l-1] and
        open[l] > close[l] and
        open[l] - close[l-1] >= -0.2 * atr[l] and
        close[l] < open[l-1]):
        return 1

    # Bullish engulfing
    elif (bodydiff[l] > bodydiffmin and
          bodydiff[l-1] > bodydiffmin and
          open[l-1] > close[l-1] and
          open[l] < close[l] and
          open[l] - close[l-1] <= 0.2 * atr[l] and
          close[l] > open[l-1]):
        return 2

    return 0

def isStar(l):
    bodydiffmin = 0.35 * atr[l]

    highdiff[l] = high[l] - max(open[l], close[l])
    lowdiff[l] = min(open[l], close[l]) - low[l]
    bodydiff[l] = abs(open[l] - close[l])

    if bodydiff[l] < 0.1 * atr[l]:
        bodydiff[l] = 0.1 * atr[l]

    ratio1[l] = highdiff[l] / bodydiff[l]
    ratio2[l] = lowdiff[l] / bodydiff[l]

    # Bearish shooting star
    if ratio1[l] > 1.5 and lowdiff[l] < 0.3 * highdiff[l] and bodydiff[l] > bodydiffmin:
        return 1

    # Bullish hammer
    if ratio2[l] > 1.5 and highdiff[l] < 0.3 * lowdiff[l] and bodydiff[l] > bodydiffmin:
        return 2

    return 0

In [189]:
def closeResistance(l, levels):
    if not levels:
        return 0

    lim = 1 * atr[l]
    level = min(levels, key=lambda x: abs(x - high[l]))

    return (
        abs(high[l] - level) <= lim and
        min(open[l], close[l]) < level and
        low[l] < level)


def closeSupport(l, levels):
    if not levels:
        return 0

    lim = 1.5 * atr[l]
    level = min(levels, key=lambda x: abs(x - low[l]))

    return (
        abs(low[l] - level) <= lim and
        max(open[l], close[l]) > level and
        high[l] > level)


In [190]:
n1 = 2
n2 = 2
backCandles = 30

signal = [0] * length

for row in range(backCandles, length - n2):
    ss, rr = [], []

    for subrow in range(row - backCandles + n1, row + 1):
        if support(df, subrow, n1, n2):
            ss.append(low[subrow])
        if resistance(df, subrow, n1, n2):
            rr.append(high[subrow])


    if (isEngulfing(row) == 1 or isStar(row) == 1) and (closeResistance(row, rr) or len(rr) >= 1):
        signal[row] = 1  # sell

    elif (isEngulfing(row) == 2 or isStar(row) == 2) and (closeSupport(row, ss) or len(ss) >= 1):
        signal[row] = 2  # buy



In [191]:
df['signal']=signal
df[df['signal']==2].count()


,0
Local time,3
open,3
high,3
low,3
close,3
volume,3
ATR,3
signal,3


In [192]:
import os

df.columns = ['Local time', 'Open', 'High', 'Low', 'Close', 'Volume', 'ATR', 'signal']
df['Local time'] = pd.to_datetime(df['Local time'])
df.set_index('Local time', inplace=True, drop=False)

df
curr_dir = os.getcwd()
df.to_excel(os.path.join(curr_dir, "signal.xlsx"), index=False)


In [193]:
!pip install backtesting

In [194]:
from backtesting import Strategy, Backtest
from backtesting import set_bokeh_output

set_bokeh_output(notebook=False)

def SIGNAL():
    return df.signal

In [195]:
class MyCandlesStrat(Strategy):
    def init(self):
        self.signal1 = self.I(SIGNAL)
        self.atr = self.data.ATR

    def next(self):
        atr = self.atr[-1]

        if self.signal1[-1] == 2 and not self.position:
            self.buy(
                sl=self.data.Close[-1] - atr,
                tp=self.data.Close[-1] + 1.5 * atr
            )

        elif self.signal1[-1] == 1 and not self.position:
            self.sell(
                sl=self.data.Close[-1] + atr,
                tp=self.data.Close[-1] - 1.5 * atr
            )



In [196]:
bt = Backtest(df, MyCandlesStrat, cash=10_000, commission=.002)
stat = bt.run()
stat



Backtest.run:   0%|          | 0/270 [00:00<?, ?bar/s]

/tmp/ipykernel_26845/2907976022.py:2: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stat = bt.run()


,0
Start,2025-03-17 00:00:00
End,2026-04-14 00:00:00
Duration,393 days 00:00:00
Exposure Time [%],12.915129
Equity Final [$],9628.847779
Equity Peak [$],10152.092
Commissions [$],155.933643
Return [%],-3.711522
Buy & Hold Return [%],20.948598
Return (Ann.) [%],-3.455856


In [197]:
from backtesting import set_bokeh_output
from bokeh.io import output_notebook

output_notebook()
set_bokeh_output(notebook=True)
bt.plot()

GridPlot(id='p8747', ...)